In [5]:
import org.eclipse.jgit.api.Git
import org.eclipse.jgit.revwalk.RevCommit
import org.eclipse.jgit.storage.file.FileRepositoryBuilder
import kotlin.io.path.Path

val project = "SYNCOPE"
val gitPath = "/home/francesco/Documents/Università/ISW/Progetto/60/sources/${project.toLowerCase()}/.git"
val path = "/home/francesco/Documents/Università/ISW/Progetto/60/data/$project"
val git = Git(FileRepositoryBuilder()
                .setGitDir(Path(gitPath).toFile())
                .build())
val commits = ArrayList<RevCommit>();
git.log().call().forEach { commits.add(it) }

In [ ]:
val pattern = "$project-\\d+".toRegex()
val mapping = commits.groupBy { it -> pattern.find(it.fullMessage)?.value }
println("${commits.size} ${mapping.size}")

In [17]:
%use dataframe
val mappingFrame = dataFrameOf(
    "issueKey" to mapping.keys.toL,
    "commits" to mapping.values.map { commits ->
        commits.map { "${it.name.substring(0..6)} (${it.authorIdent.`when`})" }.joinToString(",")
    }
)
mappingFrame


issueKey,commits
SYNCOPE-1883,426477b (Thu May 15 12:27:55 CEST 2025)
SYNCOPE-1882,ae1b410 (Thu May 15 11:14:40 CEST 202...
SYNCOPE-1880,201dddb (Tue May 13 13:41:07 CEST 2025)
SYNCOPE-1881,2510cbe (Tue May 13 12:09:49 CEST 2025)
SYNCOPE-1878,bd74b2a (Tue May 13 10:49:59 CEST 202...
SYNCOPE-1879,80665a9 (Mon May 12 14:34:40 CEST 2025)
SYNCOPE-1877,25fe08c (Fri May 09 14:58:28 CEST 2025)
SYNCOPE-1876,ceab2ea (Wed May 07 09:37:54 CEST 2025)
SYNCOPE-1874,3d6a9c7 (Mon May 05 20:30:30 CEST 2025)
SYNCOPE-1875,6f1005b (Wed Apr 30 13:16:38 CEST 2025)


In [8]:
val tags = git.tagList().call()
    .map { Pair(it.name, git.getRepository().parseCommit(it.objectId).authorIdent.`when`) }
    .map { it -> Pair(it.first.replace("refs/tags/bookkeeper-", ""), java.text.SimpleDateFormat("yyyy-MM-dd").format(it.second)) }
    .sortedBy { it -> it.second }
    .toList()
val releases = releaseApi.getLocal()
    .sortedBy { it -> it.releaseDate }

In [6]:
val issues = issueApi.issues
    .sortedBy { it.created }
    .toList()

In [7]:
issues.filter { it.affectedVersions.isNotEmpty() }.count()

99

In [8]:
val issuesFrame = dataFrameOf(
    "key" to issues.map { it.key },
    "created" to issues.map { it.created },
    "affectedVersions" to issues.map { it.affectedVersions.map { it -> if (it == null) "null" else it.name }.joinToString(",") },
    "fixVersion" to issues.map { it.fixVersion.name },
    "commits" to issues.map { it.commits.map { it -> it.name.substring(0..5) }.joinToString(",") }
)
issuesFrame


key,created,affectedVersions,fixVersion,commits
BOOKKEEPER-5,2011-02-21T20:55:47,,4.0.0,00f25f
BOOKKEEPER-1,2011-04-26T17:19:10,,4.0.0,"266b1c,9ea30d"
BOOKKEEPER-18,2011-05-16T16:40:33,,4.0.0,da1df6
BOOKKEEPER-19,2011-05-18T12:55:23,,4.0.0,8ee32a
BOOKKEEPER-22,2011-06-01T11:31:09,,4.0.0,267ed9
BOOKKEEPER-27,2011-06-29T11:11:34,,4.0.0,3e9d59
BOOKKEEPER-28,2011-06-29T12:33:42,,4.0.0,afdd3d
BOOKKEEPER-29,2011-06-29T15:35:13,,4.0.0,"40b13a,de1c3b"
BOOKKEEPER-40,2011-08-08T11:29,,4.1.0,0d43f4
BOOKKEEPER-55,2011-08-25T10:52:23,4.0.0,4.2.0,a27b64


In [19]:
val all = CsvJavaClassApi(path + "/classes.csv").getLocal()
val methods = CsvJavaMethodApi(path + "/lbl-methods.csv", all)
    .getLocal()
val classMethodMapping = methods.groupBy { it.javaClass.commit }

In [20]:
val classes = methods.map { it.javaClass }.toList()
classes.size

95064

In [23]:
%use dataframe
classes.sortedBy { it.time }
val classesFrame = dataFrameOf(
    "name" to classes.map { it.name },
    "commit" to classes.map { it.commit },
    "time" to classes.map { it.time }
)
classesFrame


name,commit,time
UserDataBinder,e4d85752fa677b7255d3e011b6d5bc4d27ae2e10,2012-04-16T15:10:59
PasswordPolicyTO,851cdd64e3b4f27d542c65449d50d716c3acea3c,2012-04-18T15:52:29
ResourceController,630dc73548ce378e95acb50e122b612eecc2988f,2012-04-18T08:31:23
TaskTest,4305a1a0e41d29a0251e09eb6880cc5c4a1a4efb,2012-04-04T13:51:56
EncryptionUtil,add292219ddecb4099a01b4721c1ad6ff725bbc0,2011-04-14T13:10:44
SearchMode,34453842397fc2fa9228aca0c015a87ac5134796,2011-01-19T09:19:39
QueryUtils,a87430ff351cbaefdd20ec0afaff1a8c908b505d,2010-07-22T10:00:16
WorkflowFormPropertyTO,851cdd64e3b4f27d542c65449d50d716c3acea3c,2012-04-18T15:52:29
ResourceDAOImpl,a7e14552ab944a08142942cd51707f61c0c53c13,2010-12-02T13:05:47
PropagationTO,047fc4764ef9cd813d79103853713543600b7078,2012-03-16T15:10:26


In [26]:
import io.github.francescodonnini.model.JavaClass
import io.github.francescodonnini.model.Release
import java.time.LocalDate

val releaseClassMapping = mutableMapOf<Release, MutableList<JavaClass>>()
var prev = LocalDate.MIN
for (r in releases) {
    val curr = r.releaseDate
    classes
        .filter { it -> !it.time.toLocalDate().isBefore(prev) }
        .filter { it -> !it.time.toLocalDate().isAfter(curr) }
        .forEach { it -> releaseClassMapping.getOrPut(r) { mutableListOf<JavaClass>() }.add(it) }
    prev = curr
}

In [27]:
releaseClassMapping.entries.stream().mapToInt { (k, v) -> v.size }.average().orElse(0.0)

3271.4666666666667

Let's create a DataFrame to show the releases information using the existing `releases` variable which contains the release data we need.

In [6]:
%use dataframe
val releasesFrame = dataFrameOf(
    "version" to releases.map { it.name },
    "releaseDate" to releases.map { it.releaseDate },
    "id" to releases.map { it.id }
)
releasesFrame


version,releaseDate,id
1.0.0-incubating,2012-08-06,12320044
1.0.1-incubating,2012-08-29,12322540
1.0.3-incubating,2012-09-30,12323346
1.0.2-incubating,2012-10-02,12323254
1.0.4,2012-12-10,12323474
1.0.5,2013-01-23,12323749
1.0.6,2013-02-27,12324001
1.0.7,2013-03-26,12324104
1.1.0,2013-04-05,12322504
1.0.8,2013-04-18,12324279


In [9]:
import io.github.francescodonnini.model.Issue
import org.eclipse.jgit.diff.DiffFormatter
import org.eclipse.jgit.util.io.DisabledOutputStream

val df = DiffFormatter(DisabledOutputStream.INSTANCE)
df.setRepository(git.repository)
df.setDetectRenames(true)
val issue = issues[6]
println(issue)

Issue[affectedVersions=[], created=2011-06-29T12:33:42, fixVersion=(12317843, 4.0.0, 2011-12-07), openingVersion=(12317843, 4.0.0, 2011-12-07), commits=[commit afdd3dd1e7e06f97e29ff5bcaa2001f81f6d9210 1314693271 -----sp], key=BOOKKEEPER-28, project=Bookkeeper]


In [10]:
import io.github.francescodonnini.model.LineRange
import org.eclipse.jgit.diff.Edit

issue.commits.forEach { commit ->
    println("parent:\t${commit.parents[0].name}")
    println("commit:\t${commit.name}")
    df.scan(commit.parents[0].tree, commit.tree).forEach { diff ->
        println(diff)
        val path = diff.newPath
        println(path)
        val editList = df.toFileHeader(diff).toEditList()
        for (edit in editList) {
            println(edit)
            println("A\t${edit.beginA} ${edit.endA}")
            println("B\t${edit.beginB} ${edit.endB}")
            when (edit.type) {
                Edit.Type.DELETE, Edit.Type.INSERT, Edit.Type.REPLACE -> {
                    edit.extendB()
                    val range = LineRange(edit.beginB, edit.endB)
                    val touched = methods[commit.name]
                        ?.filter { it.javaClass.path.toString() == path }
                        ?.filter { it.range.intersects(range) }
                    if (touched == null || touched.isEmpty())
                        println("no method touched")
                    else
                        println("method touched: ${touched}")
                }
                else -> { println("${edit.type} not supported yet") }
            }
        }
    }
}


org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[10], line 20, column 35: Unresolved reference: methods
at Cell In[10], line 21, column 36: Unresolved reference: it
at Cell In[10], line 22, column 36: Unresolved reference: it

In [15]:
issues[0].key

BOOKKEEPER-5

In [17]:
import io.github.francescodonnini.model.LineRange
import org.eclipse.jgit.api.Git
import org.eclipse.jgit.diff.DiffFormatter
import org.eclipse.jgit.diff.Edit
import org.eclipse.jgit.revwalk.RevCommit
import org.eclipse.jgit.storage.file.FileRepositoryBuilder
import org.eclipse.jgit.util.io.DisabledOutputStream
import kotlin.io.path.Path


val git = Git(
    FileRepositoryBuilder()
        .setGitDir(Path("/home/francesco/Documents/Università/ISW/Progetto/60/sources/bookkeeper/.git").toFile())
        .build()
)
val issue = issues.filter { it -> it.key.startsWith("BOOKKEEPER-74") }.first()
val df = DiffFormatter(DisabledOutputStream.INSTANCE)
df.setRepository(git.repository)
df.setDetectRenames(true)
issue.commits
    .filter { it.parents.isNotEmpty() && it.parents[0].tree != null }
    .forEach { commit ->
        println("parent:\t${commit.parents[0].name}")
        println("commit:\t${commit.name}")
        df.scan(commit.parents[0].tree, commit.tree).forEach { diff ->
            println(diff)
            val path = diff.newPath
            println(path)
            val editList = df.toFileHeader(diff).toEditList()
            for (edit in editList) {
                println(edit)
                println("A\t${edit.beginA} ${edit.endA}")
                println("B\t${edit.beginB} ${edit.endB}")
            }
    }
}


parent:	9366322bfe3461a44a8b0444e66cd774ea1ac7d8
commit:	76818a6aadd4aa40a45aca6f5b0acba79cd42515
DiffEntry[MODIFY CHANGES.txt]
CHANGES.txt
INSERT(56-56,56-58)
A	56 56
B	56 58
DiffEntry[MODIFY hedwig-server/src/main/java/org/apache/hedwig/server/persistence/BookkeeperPersistenceManager.java]
hedwig-server/src/main/java/org/apache/hedwig/server/persistence/BookkeeperPersistenceManager.java
INSERT(26-26,26-27)
A	26 26
B	26 27
INSERT(79-79,80-81)
A	79 79
B	80 81
INSERT(127-127,129-133)
A	127 127
B	129 133
INSERT(150-150,156-157)
A	150 150
B	156 157
INSERT(361-361,368-374)
A	361 361
B	368 374
INSERT(385-385,398-412)
A	385 385
B	398 412
DiffEntry[MODIFY hedwig-server/src/test/java/org/apache/hedwig/server/integration/TestHedwigHub.java]
hedwig-server/src/test/java/org/apache/hedwig/server/integration/TestHedwigHub.java
REPLACE(256-257,256-257)
A	256 257
B	256 257
REPLACE(258-259,258-259)
A	258 259
B	258 259
REPLACE(260-273,260-262)
A	260 273
B	260 262
REPLACE(389-390,378-379)
A	389 390
B	37